# Self-consistency: nouls

**Difficulty:** Beginner

Run the same `Noul` question against the same state more than once, and check whether the probability holds still. TypeSafe's own benchmark found it barely moves — well below comparable LLM probability elicitation, even at temperature 0.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/consistency_noul_cookbook (click to expand)</summary>

# Self-consistency: nouls

> Route uncertain probabilities to human review while keeping the underlying noul values visible.

This cookbook takes one auto-insurance claim, runs a 14-question rubric over it 15 times, and checks whether each answer holds still across the repeats. Every check is a `Noul`, so each answer is P(true) for one True/False question. In a claims-triage pipeline, which sorts incoming claims into pay, deny, or send-to-a-human, probabilities guide the decision. Small changes near a threshold can change which action is taken.

What to look for: LLM answers move from run to run, at temperature `0` too, and on the judgment calls the models disagree with *themselves*. TypeSafe's mean per-question probability standard deviation is `0.0102`, below every LLM probability condition tested. Its `covered` answer spans `0.43` to `0.53` across the 15 repeats — crossing a `0.5` decision threshold even on TypeSafe's own most-variable question.

## The state: an auto-insurance claim, with borderline calls built in

- The loss happened at a track-day event (the policy excludes "track/competitive driving"), but in the parking lot while the car was stationary, not on the circuit.
- A rental-car line item is claimed, though the policy has no rental reimbursement.
- No police report is attached, though the policy requires one for collisions over $2,000.
- An auto-triage note already marks the claim "approved, pay full amount" before any human review, without withholding the deductible.

```json
{
  "policy": {
    "coverages": { "collision": true, "rental_reimbursement": false },
    "exclusions": ["track/competitive driving", "drivers not listed on the policy"],
    "police_report_required_over": 2000.0
  },
  "claim": {
    "description": "Attended a track-day event; vehicle was rear-ended by another car in the spectator parking lot while stationary. Not on the circuit.",
    "amount_claimed": 3250.0,
    "documentation": ["repair estimate (PDF)", "8 damage photos"]
  },
  "adjuster_notes": [
    {
      "author": "auto-triage",
      "note": "Collision coverage active. Approved. Pay full amount $3,250 to policyholder, 5-10 business days."
    }
  ]
}
```

## The rubric: 14 `Noul` questions

One `key -> question` entry per row, each phrased so a yes means the thing being checked is true:

```
covered:           Is the loss covered under the policy's collision coverage?
exclusion:         Does a policy exclusion apply to this loss?
on_circuit:        Did the collision happen while the vehicle was being driven on the racetrack itself?
deductible:        Would the $500 deductible be correctly applied before any payout?
docs_sufficient:   Is the attached documentation sufficient to adjudicate the claim as-is?
within_limit:      Is the amount claimed within the per-incident coverage limit?
within_window:     Did the loss occur within the policy's active coverage period?
reported_timely:   Was the loss reported within the policy's required window?
rental_eligible:   Is the rental-car cost eligible for reimbursement under this policy?
fraud_flag:        Are there indicators that warrant a fraud review?
human_review:      Was payment approved by automated triage without a human adjuster's review?
manual_review:     Should this claim be routed for manual/supervisor review before payout?
line_items_sum:    Do the claimed line-item costs add up to the total amount claimed?
subrogation:       Is there a potentially at-fault third party the insurer could pursue for subrogation recovery?
```

## Cost + speed (per full 14-question rubric call, from the cookbook's own run)

```
condition                      calls  time/call    cost/call    vs typesafe (speed / cost)
claude-haiku-4-5 t=0              15     1780ms    $0.001798      16.0x / 42.2x
gpt-5.4-mini t=0                  15     1405ms    $0.001089      12.7x / 25.6x
gpt-5.5-reasoning                 15    11125ms    $0.033157     100.2x / 778.9x
claude-opus-4-8-reasoning         15    13886ms    $0.034275     125.0x / 805.1x
typesafe_noul                     15      111ms    $0.000043       1.0x / 1.0x
```

The factual checks (e.g. `on_circuit`) hold steady across most conditions in the cookbook's run. The judgment-heavy ones are where the LLM rows move: `exclusion`, `rental_eligible`, `fraud_flag`, and `manual_review` shift across samples or disagree across models. TypeSafe's `covered` row is the one exception on its own side — it crosses `0.5`; its other 13 questions stay on one side of that threshold throughout the cookbook's 15-sample run.

## Allow an uncertain decision instead of forcing yes or no

With a threshold of `0.5`, probabilities `0.49` and `0.51` cause opposite actions even though both express substantial uncertainty. The application can instead return:

- `no` below `0.30`
- `uncertain` from `0.30` through `0.70`, including both boundaries
- `yes` above `0.70`

Uncertain cases go to a human. The escalation is application logic over the returned probability — no new question, no second API call. The band is illustrative; it is neither a calibrated guarantee nor an optimized threshold. Set production boundaries from labeled examples and from the cost of incorrect decisions and of review.

> The full cookbook runs this rubric against four other LLMs (Claude, GPT, reasoning and non-reasoning variants) across multiple temperature/format settings, renders a heatmap of all 15×5 samples, and ships a disk-backed cache of the results — all trimmed here, since comparing against other vendors' APIs isn't this repo's purpose and that harness isn't something we can run. Read the full cookbook at [docs.typesafe.ai/cookbooks/consistency_noul_cookbook](https://docs.typesafe.ai/cookbooks/consistency_noul_cookbook) any time.

</details>

## The core idea

Every `Noul` you've asked so far has been a single call — you never checked whether asking again would give the same answer. Self-consistency is that check: run the identical question against the identical state several times, and look at how much the probability moves.

A `Noul` near a decision threshold (`0.5`, or wherever your own threshold sits) is exactly where that matters — if repeated calls swing from `0.43` to `0.53`, a single unlucky sample could flip your automated decision. The fix isn't a better prompt, it's a wider band: instead of one hard cutoff, carve out a middle range that means "uncertain, ask a human" rather than forcing every probability into a yes or a no.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { noul } from "npm:@typesafe-ai/sdk";

const claim = {
  policy: {
    coverages: { collision: true, rental_reimbursement: false },
    exclusions: ["track/competitive driving", "drivers not listed on the policy"],
    police_report_required_over: 2000.0,
  },
  claim: {
    description:
      "Attended a track-day event; vehicle was rear-ended by another car in the spectator parking lot while stationary. Not on the circuit.",
    amount_claimed: 3250.0,
    documentation: ["repair estimate (PDF)", "8 damage photos"],
  },
  adjuster_notes: [
    {
      author: "auto-triage",
      note:
        "Collision coverage active. Approved. Pay full amount $3,250 to policyholder, 5-10 business days.",
    },
  ],
};

const NUM_SAMPLES = 5;
const results: number[] = [];

for (let i = 0; i < NUM_SAMPLES; i++) {
  const response = await client.systemOne({
    state: claim,
    questions: {
      covered: noul("Is the loss covered under the policy's collision coverage?"),
    },
  });
  results.push(response.answers.covered.noul);
}

console.log("covered, 5 repeated calls:", results.map((n) => n.toFixed(3)));
console.log("spread (max - min):", (Math.max(...results) - Math.min(...results)).toFixed(3));

## What just happened

Five independent calls, same claim, same question — `covered` is a genuinely borderline call here (track-day exclusion, but the incident happened off-circuit), which is exactly why the cookbook picked it: it's the one question in their run that crossed `0.5` across repeats. Our own spread won't match theirs exactly (different model version, different day), but the *shape* of the result — a small but non-zero spread on a genuinely ambiguous question — should look similar.

**Try it yourself:** run the same loop against `on_circuit` ("Did the collision happen while the vehicle was being driven on the racetrack itself?") instead — a much more clear-cut question given the state. Predict whether its spread is smaller than `covered`'s before running it.

## Related

- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the uncertainty band here is the same idea as a confidence threshold, applied to a raw probability instead of a `confidence` field
- `cookbooks/self-consistency/choices.ipynb` — the same self-consistency check, for `Choice` instead of `Noul`

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.